# Etapa 2 — EDA + Construcción de Features (Pipeline_Integrado principal)

Construye todos los bloques de features hospitalarias a partir del `grd_filtrado.parquet`:

1. **Tradicionales** (6): egresos/año, estancia media/mediana, peso GRD, severidad, mortalidad.
2. **Diversidad** (2): entropía de Shannon de GRD, comorbilidades promedio.
3. **CMA** (2): tasa CMA, peso medio CMA.
4. **Casuística clínica**: vectores de capítulos CIE-10, secciones CIE-9-MC y Top-20 GRDs nacionales (→ PCA en etapa 3).
5. **Extendidas / agregadas** (18): demográficas, mix de ingreso, tipo de alta, procedencia, pabellón, obstétricas y estancia.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert np.__version__.startswith('1.'), 'NumPy debe ser 1.x'

from src.etl.casuistica import Constructor_Features
from src.etl.cie_mappers import Mapeador_CIE10, Mapeador_CIE9
from src.etl.features_extendidas import Constructor_Features_Extendidas
from src.utils.io import write_parquet, read_parquet, TABLES_DIR, FIGURES_DIR
from src.utils.validators import (
    Validador_CIE10, Validador_CIE9, Validador_Correlaciones,
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', font_scale=1.05)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2.1 Carga de la base GRD filtrada

In [2]:
df_grd = read_parquet('grd_filtrado')
print(f'Egresos: {len(df_grd):,} | columnas: {df_grd.shape[1]}')
print(f"Hospitales (raw): {df_grd['COD_HOSPITAL'].nunique()}")
print('MODALIDAD:', df_grd['MODALIDAD'].value_counts().to_dict())

Egresos: 5,808,515 | columnas: 36
Hospitales (raw): 72
MODALIDAD: {'HOSPITALIZACION': 4918647, 'CMA': 889868}


## 2.2 Mapeadores CIE y validación de cobertura

In [3]:
df_cie10 = pd.read_excel(ROOT / 'insumos' / 'maestras' / 'CIE-10.xlsx')
df_cie9 = pd.read_excel(ROOT / 'insumos' / 'maestras' / 'CIE-9 .xlsx')
mapper_cie10 = Mapeador_CIE10(df_maestra=df_cie10)
mapper_cie9 = Mapeador_CIE9(df_maestra=df_cie9)
mapper_cie10.construir(); mapper_cie9.construir()

val10 = Validador_CIE10(df_grd=df_grd, mapper=mapper_cie10)
cobertura_cie10 = val10.validar_por_anio()
print('Cobertura CIE-10 por año:')
print(cobertura_cie10.to_string(index=False))
val9 = Validador_CIE9(df_grd=df_grd, mapper=mapper_cie9)
cobertura_cie9 = val9.validar_por_anio()
print('\nCobertura CIE-9-MC por año:')
print(cobertura_cie9.to_string(index=False))

Cobertura CIE-10 por año:
 anio  n_total  n_mapeados  cobertura
 2019  1004449      996803     0.9924
 2020   702275      700246     0.9971
 2021   713978      713754     0.9997
 2022   778073      777925     0.9998
 2023   846043      846011     1.0000
 2024   873772      873742     1.0000



Cobertura CIE-9-MC por año:
 anio  n_total  n_mapeados  cobertura
 2019   993517      993516     1.0000
 2020   695545      695545     1.0000
 2021   713836      713833     1.0000
 2022   777938      777934     1.0000
 2023   845885      845874     1.0000
 2024   873680      873667     1.0000


## 2.3 Elegibilidad de hospitales (≥3 años, ≥500 egresos hosp/año)

In [4]:
constructor = Constructor_Features(
    df_grd=df_grd, mapper_cie10=mapper_cie10, mapper_cie9=mapper_cie9,
)
elegibles = constructor.hospitales_elegibles()
descartados = constructor.descartados()
print(f'Hospitales elegibles:   {len(elegibles)}')
print(f'Hospitales descartados: {len(descartados)}')
if len(descartados):
    descartados.to_csv(TABLES_DIR / 'hospitales_descartados.csv', index=False)

Hospitales elegibles:   65
Hospitales descartados: 7


## 2.4 Features tradicionales, diversidad y CMA

In [5]:
f_trad = constructor.features_tradicionales()
f_div = constructor.features_diversidad()
f_cma = constructor.features_cma()
print('Tradicionales:', f_trad.shape, '| Diversidad:', f_div.shape, '| CMA:', f_cma.shape)

write_parquet(f_trad, 'features_tradicionales',
    expected_cols=['COD_HOSPITAL', 'egresos_por_anio', 'estancia_media',
        'estancia_mediana', 'peso_medio_grd', 'severidad_media', 'mortalidad_media'])
write_parquet(f_div, 'features_diversidad',
    expected_cols=['COD_HOSPITAL', 'entropia_grd', 'comorbilidades_promedio'])
write_parquet(f_cma, 'features_cma',
    expected_cols=['COD_HOSPITAL', 'tasa_cma', 'peso_medio_cma'])

Tradicionales: (65, 7) | Diversidad: (65, 3) | CMA: (65, 3)


PosixPath('/home/reinaldo/tesis-pregrado/data/processed/features_cma.parquet')

## 2.5 Vectores de casuística clínica (CIE-10 / CIE-9-MC / Top-20)

In [6]:
v_cap_principal = constructor.vector_capitulos('principal')
v_cap_ponderado = constructor.vector_capitulos('ponderado')
v_secciones = constructor.vector_secciones()
top20 = constructor.top20_grds_nacionales()
v_top20 = constructor.vector_top20(top20)
top20_resumen = constructor.top20_grds_resumen(top20)

print('Capítulos principal:', v_cap_principal.shape, '| ponderado:', v_cap_ponderado.shape)
print('Secciones:', v_secciones.shape, '| Top-20:', v_top20.shape)

v_capitulos_long = pd.concat([v_cap_principal, v_cap_ponderado], ignore_index=True)
write_parquet(v_capitulos_long, 'casuistica_capitulos', expected_cols=['COD_HOSPITAL', 'variante'])
write_parquet(v_secciones, 'casuistica_procedimientos', expected_cols=['COD_HOSPITAL'])
write_parquet(v_top20, 'casuistica_top20_grds', expected_cols=['COD_HOSPITAL'])
top20_resumen.to_csv(TABLES_DIR / 'top20_grds_nacionales.csv', index=False)

Capítulos principal: (65, 25) | ponderado: (65, 25)
Secciones: (65, 20) | Top-20: (65, 22)


## 2.6 Features extendidas / agregadas (variables nuevas del pipeline)

Estas 18 features describen el **perfil funcional** del hospital más allá del GRD básico. Se calculan solo sobre HOSPITALIZACIÓN y hospitales elegibles.

In [7]:
constructor_ext = Constructor_Features_Extendidas(
    df_grd=df_grd, hospitales_elegibles=elegibles,
)
features_ext = constructor_ext.construir_todas()
print('Features extendidas:', features_ext.shape)
print('Columnas:', [c for c in features_ext.columns if c != 'COD_HOSPITAL'])
write_parquet(features_ext, 'features_extendidas', expected_cols=['COD_HOSPITAL'])

Features extendidas: (65, 19)
Columnas: ['pct_pediatrico', 'pct_geriatrico', 'pct_femenino_fertil', 'edad_mediana', 'pct_urgencia', 'pct_programada', 'pct_obstetrica_ingreso', 'pct_alta_domicilio', 'pct_alta_fallecido', 'pct_alta_traslado', 'pct_origen_emergencia', 'pct_origen_referencia', 'pct_uso_pabellon', 'pabellones_promedio', 'tasa_partos', 'tasa_prematurez', 'cv_estancia', 'pct_estancia_larga']


PosixPath('/home/reinaldo/tesis-pregrado/data/processed/features_extendidas.parquet')

## 2.7 Validación de correlaciones (entropía vs volumen)

In [8]:
matriz_trad = (f_trad.merge(f_div, on='COD_HOSPITAL')
                     .merge(f_cma, on='COD_HOSPITAL'))
val_corr = Validador_Correlaciones(matriz_integrada=matriz_trad)
df_corr = val_corr.correlaciones_spearman()
df_corr.to_csv(TABLES_DIR / 'correlaciones_features.csv', index=False)
adv = val_corr.advertencia_entropia_volumen()
print('Advertencia entropía↔volumen:', adv or '(sin advertencia, corr<0.85)')

Advertencia entropía↔volumen: (sin advertencia, corr<0.85)


## 2.8 EDA: distribución de las features agregadas por hospital

In [9]:
cols_plot = ['pct_pediatrico', 'pct_geriatrico', 'pct_urgencia', 'pct_programada',
             'tasa_partos', 'pct_uso_pabellon', 'edad_mediana', 'cv_estancia']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), cols_plot):
    ax.hist(features_ext[col].dropna(), bins=20, color='steelblue', edgecolor='black')
    ax.set_title(col, fontsize=10)
fig.suptitle('Distribución de features agregadas (n=%d hospitales)' % len(features_ext), y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_features_extendidas.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figura: eda_features_extendidas.png')

Figura: eda_features_extendidas.png


## 2.9 Resumen Etapa 2

In [10]:
print('=' * 60)
print('RESUMEN ETAPA 2 — features')
print('=' * 60)
print(f'Hospitales elegibles:  {len(elegibles)}')
print(f'Tradicionales:         {f_trad.shape[1]-1}')
print(f'Diversidad:            {f_div.shape[1]-1}')
print(f'CMA:                   {f_cma.shape[1]-1}')
print(f'Extendidas/agregadas:  {features_ext.shape[1]-1}')
print(f'Cobertura CIE-10 mín:  {cobertura_cie10["cobertura"].min():.1%}')
print(f'Cobertura CIE-9 mín:   {cobertura_cie9["cobertura"].min():.1%}')
print('\nSiguiente: 03_clustering.ipynb')

RESUMEN ETAPA 2 — features
Hospitales elegibles:  65
Tradicionales:         6
Diversidad:            2
CMA:                   2
Extendidas/agregadas:  18
Cobertura CIE-10 mín:  99.2%
Cobertura CIE-9 mín:   100.0%

Siguiente: 03_clustering.ipynb
